# Modeling dan validasi

Notebook ini menjalankan workflow yang dimodulkan pada `scripts/train.py`. Pipeline menggunakan `ColumnTransformer` untuk preprocessing dan `Pipeline` untuk menggabungkan preprocessing dengan CatBoost. Validasi memakai Macro F1 pada holdout stratified; neural network tidak digunakan.

Gunakan ID eksperimen baru agar laporan lama tidak tertimpa. Kandidat di bawah hanya contoh konfigurasi CatBoost yang dapat dibandingkan. Jalankan sel dengan sadar karena pencarian membutuhkan waktu.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'train.py').exists():
    ROOT = ROOT.parent
EXPERIMENT_ID = 'notebook_modeling_20261002'
CANDIDATES = [
    {'depth': 6, 'learning_rate': 0.05, 'iterations': 700, 'l2_leaf_reg': 5},
    {'depth': 7, 'learning_rate': 0.05, 'iterations': 700, 'l2_leaf_reg': 5},
]
command = [sys.executable, str(ROOT / 'scripts' / 'train.py'), '--seed', '42', '--force-search', '--experiment-id', EXPERIMENT_ID]
for candidate in CANDIDATES:
    command.extend(['--candidate', json.dumps(candidate)])
subprocess.run(command, cwd=ROOT, check=True)


## Hasil validasi

Laporan memuat Macro F1, konfigurasi terbaik, metrik per kelas, dan confusion matrix dari holdout. Skor lokal ini bukan skor leaderboard Kaggle.


In [ ]:
report_path = ROOT / 'reports' / f'validation_{EXPERIMENT_ID}.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Macro F1:', report['macro_f1'])
print('Parameter terbaik:', report['best_parameters'])
print('Iterasi terbaik:', report['best_iteration'])
print('Classification report:')
display(pd.DataFrame(report['classification_report']).T if isinstance(report.get('classification_report'), dict) else report.get('classification_report'))
print('Confusion matrix:')
display(pd.DataFrame(report['confusion_matrix']))


## Fit model final (opsional)

Setelah memilih validasi yang akan dipakai, jalankan sel berikut untuk melatih pipeline pada seluruh data berlabel dan menyimpannya di `models/`. Perintah memakai laporan eksperimen di atas. Submission kompetisi tetap dibuat dan diperiksa di `03_submission.ipynb`.


In [ ]:
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'train.py'), '--final', '--experiment-id', EXPERIMENT_ID], cwd=ROOT, check=True)
